# Fake.br: detecção de anomalias com Isolation Forest

Notebook autossuficiente para Colab e execução local. O modelo aprende o padrão de notícias **True (0)** e sinaliza textos com comportamento linguístico atípico. Notícias **Fake (1)** ficam fora do ajuste e da calibração; são usadas somente para avaliação exploratória.

> **Interpretação:** `anomalyScore` é um score de anomalia, não uma probabilidade de falsidade. Um alerta indica necessidade de revisão e não comprova que a notícia seja falsa.

Para usar no Colab, faça upload deste arquivo e escolha **Runtime → Run all**. O ZIP do Fake.br é baixado automaticamente para `data/` quando ainda não estiver no ambiente.

## 1. Imports e configuração do ambiente

Execute esta célula em um kernel limpo. No Colab, as bibliotecas `numpy`, `pandas`, `scikit-learn` e `matplotlib` normalmente já estão disponíveis; se o ambiente solicitar, instale-as antes de executar todas as células.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from itertools import combinations
from importlib.metadata import version
import hashlib
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay,
)

In [ ]:
RANDOM_STATE = 42
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("Python:", platform.python_version())
print({name: version(name) for name in ["numpy", "pandas", "scikit-learn", "matplotlib", "ipykernel"]})

## 3. Carregamento dos dados

Mesmo ZIP e revisão fixa do notebook original. Textos completos e metadados são
carregados para auditoria; o truncamento acontece ANTES da extração usada nos
modelos. Nenhuma notícia é excluída por comprimento.

In [ ]:
CORPUS_REVISION = "780f5516c4ae070761632d98ac3368f3ded09d35"
CORPUS_URL = f"https://codeload.github.com/roneysco/Fake.br-Corpus/zip/{CORPUS_REVISION}"
projectFolder = Path.cwd()
dataFolder = projectFolder / "data"
dataFolder.mkdir(exist_ok=True)
archivePath = dataFolder / f"Fake.br-Corpus-{CORPUS_REVISION}.zip"

if not archivePath.exists():
    temporaryPath = archivePath.with_suffix(".download")
    with urlopen(CORPUS_URL, timeout=120) as response, temporaryPath.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    with ZipFile(temporaryPath) as archive:
        assert archive.testzip() is None, "ZIP corrompido; refaça o download."
    temporaryPath.replace(archivePath)

print("Corpus revision:", CORPUS_REVISION)
print("Archive SHA256:", hashlib.sha256(archivePath.read_bytes()).hexdigest())

In [ ]:
metadataColumns = [
    "autor", "link", "categoria", "data_publicacao",
    "num_tokens", "num_palavras", "num_types", "num_links", "num_maiusculas",
    "num_verbos", "num_verbos_subj_imp", "num_substantivos", "num_adjetivos",
    "num_adverbios", "num_verbos_modais", "num_pron_1_2_sing", "num_pron_1_plural",
    "num_pronomes", "pausalidade", "num_caracteres", "tam_medio_sentenca",
    "tam_medio_palavra", "pct_erros_ortograficos", "emotividade", "diversidade",
]

In [ ]:
def loadNewsTexts(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}/" not in name or not name.endswith(".txt"):
            continue
        articleId = Path(name).stem + ("t" if label == 0 else "")
        records.append({
            "id": articleId, "text": archive.read(name).decode("utf-8"),
            "label": label, "sourceClass": folder,
        })
    assert records, f"Nenhum texto encontrado em {folder}"
    return pd.DataFrame(records)


def loadNewsMetadata(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}-meta-information/" not in name or not name.endswith("-meta.txt"):
            continue
        values = [line.strip() for line in archive.read(name).decode("utf-8").splitlines()]
        assert len(values) == len(metadataColumns), f"Esquema inesperado em {name}: {len(values)} linhas"
        articleId = Path(name).name.removesuffix("-meta.txt") + ("t" if label == 0 else "")
        records.append({**dict(zip(metadataColumns, values)), "id": articleId, "metadataLabel": label})
    assert records, f"Nenhum metadado encontrado em {folder}"
    return pd.DataFrame(records)

In [ ]:
with ZipFile(archivePath) as archive:
    textsFrame = pd.concat([
        loadNewsTexts(archive, "fake", 1), loadNewsTexts(archive, "true", 0),
    ], ignore_index=True)
    metadataFrame = pd.concat([
        loadNewsMetadata(archive, "fake", 1), loadNewsMetadata(archive, "true", 0),
    ], ignore_index=True)

assert textsFrame["id"].is_unique and metadataFrame["id"].is_unique
assert set(textsFrame["id"]) == set(metadataFrame["id"]), "Texto/metadados sem correspondência"
newsFrame = textsFrame.merge(metadataFrame, on="id", validate="one_to_one", indicator=True)
assert newsFrame["_merge"].eq("both").all()
assert newsFrame["label"].eq(newsFrame["metadataLabel"]).all()
newsFrame = newsFrame.drop(columns=["_merge", "metadataLabel"])
assert newsFrame["text"].str.strip().ne("").all()
print(f"{len(newsFrame):,} notícias carregadas; textos e metadados correspondem 1:1.")

## 4. Contrato dos labels

**0 = True; 1 = Fake.** O rótulo forma as partições e permite avaliação externa.
Não entra como feature. A origem nas pastas também é verificada.

In [ ]:
labelNames = {0: "True", 1: "Fake"}
assert labelNames == {0: "True", 1: "Fake"}
assert set(newsFrame["label"].unique()) == {0, 1}
assert newsFrame.loc[newsFrame["label"].eq(0), "sourceClass"].eq("true").all()
assert newsFrame.loc[newsFrame["label"].eq(1), "sourceClass"].eq("fake").all()
display(newsFrame.groupby(["label", "sourceClass"]).size().rename("newsCount").to_frame())

## 5. Preparação dos metadados

Preservamos todos os campos brutos em `rawNewsFrame` e todas as contagens numéricas
em `newsFrame`. Valores não numéricos viram NaN com contagem explícita; nenhum
ausente é preenchido antes do treino. `tem_autor` segue a lógica do original:
ausência para string vazia, `None`, `none` ou `NULL`. Não é uma medida de credibilidade.

In [ ]:
rawNewsFrame = newsFrame.copy(deep=True)
numericMetadataColumns = metadataColumns[4:]
convertedMetadata = newsFrame[numericMetadataColumns].apply(pd.to_numeric, errors="coerce")
conversionMissing = convertedMetadata.isna().sum().rename("missingAfterNumericConversion")
display(conversionMissing.to_frame())
newsFrame[numericMetadataColumns] = convertedMetadata
newsFrame["tem_autor"] = (~newsFrame["autor"].fillna("").astype(str).str.strip().isin(
    ["", "None", "none", "NULL"]
)).astype(int)

## 6. Extração no texto efetivamente utilizado

Normalização Unicode NFKC, remoção do BOM inicial e primeiros CHARACTER_LIMIT
caracteres, incluindo espaços/pontuação. Textos menores permanecem menores, sem
preenchimento. O corte pode dividir palavras/frases. Palavras são sequências de
letras com hífen/apóstrofo interno; tokens também incluem números e pontuação.
Tipos são palavras distintas ignorando caixa. TTR = tipos/tokens; diversidade =
tipos/palavras. Maiúsculas conta palavras totalmente maiúsculas com mais de uma
letra. Links são URLs http(s) presentes no corpo. Estas definições explícitas
não pretendem reproduzir o extrator desconhecido dos metadados históricos.

As demais contagens linguísticas são marcadas ausentes na cópia de features,
não imputadas nem selecionadas pelo modelo; originais ficam em rawNewsFrame e
newsFrame. Autor permanece como metadado válido da notícia inteira.

In [ ]:
def calculateRatio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce").astype(float)
    denominator = pd.to_numeric(denominator, errors="coerce").astype(float)
    safeDenominator = denominator.where(denominator.gt(0) & np.isfinite(denominator))
    return numerator.div(safeDenominator).replace([np.inf, -np.inf], np.nan)

In [ ]:
import re
import unicodedata

CHARACTER_LIMIT = 300
numericMetadataColumns = metadataColumns[4:]
anomalyColumns = ["tem_autor", "typeTokenRatio", "linkDensity", "punctuationDensity", "uppercaseRatio", "diversidade"]
wordPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
tokenPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]", re.UNICODE)


def measureText(text, characterLimit):
    normalized = unicodedata.normalize("NFKC", text).lstrip("\ufeff")
    if characterLimit is not None:
        if not isinstance(characterLimit, int) or characterLimit < 1:
            raise ValueError("Limite deve ser inteiro positivo ou None.")
        normalized = normalized[:characterLimit]
    words = wordPattern.findall(normalized)
    return {"text": normalized, "num_palavras": len(words),
            "num_tokens": len(tokenPattern.findall(normalized)),
            "num_types": len({word.casefold() for word in words}),
            "num_links": len(re.findall(r"https?://\S+", normalized, flags=re.IGNORECASE)),
            "num_maiusculas": sum(word.isupper() and len(word) > 1 for word in words),
            "num_caracteres": len(normalized)}


def extractAnomalyFeatures(newsFrame, characterLimit=CHARACTER_LIMIT):
    featuresFrame = newsFrame.copy(deep=True)
    featuresFrame["num_palavras_original"] = newsFrame["num_palavras"]
    featuresFrame["text_original"] = newsFrame["text"]
    featuresFrame[numericMetadataColumns] = np.nan
    measurements = pd.DataFrame([measureText(text, characterLimit) for text in newsFrame["text"]], index=newsFrame.index)
    for column in measurements:
        featuresFrame[column] = measurements[column]
    featuresFrame["typeTokenRatio"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_tokens"])
    featuresFrame["diversidade"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_palavras"])
    featuresFrame["linkDensity"] = calculateRatio(featuresFrame["num_links"], featuresFrame["num_palavras"])
    featuresFrame["punctuationDensity"] = calculateRatio(featuresFrame["num_tokens"] - featuresFrame["num_palavras"], featuresFrame["num_tokens"])
    featuresFrame["uppercaseRatio"] = calculateRatio(featuresFrame["num_maiusculas"], featuresFrame["num_palavras"])
    return featuresFrame

In [ ]:
featuresFrame = extractAnomalyFeatures(newsFrame)
featuresFrame[anomalyColumns] = featuresFrame[anomalyColumns].replace([np.inf, -np.inf], np.nan)
display(featuresFrame[anomalyColumns].isna().sum().rename("missingCount").to_frame())
print("Limite de caracteres:", CHARACTER_LIMIT)
display(featuresFrame.groupby("label")[["num_palavras_original", "num_palavras", "num_caracteres"]].agg(["min", "median", "max"]))
print("Textos menores que o limite:", int(featuresFrame["num_caracteres"].lt(CHARACTER_LIMIT).sum()))

## 7. Sanity checks

As seis features utilizam o prefixo; a presença de autor vem dos metadados.
Contagens originais ficam preservadas para auditoria e não alimentam o modelo.

In [ ]:
assert "num_palavras" not in anomalyColumns
assert "label" not in anomalyColumns and "id" not in anomalyColumns
assert len(anomalyColumns) == len(set(anomalyColumns)) == 6
assert featuresFrame["id"].is_unique
assert featuresFrame["num_caracteres"].le(CHARACTER_LIMIT).all()
assert featuresFrame["num_palavras_original"].equals(newsFrame["num_palavras"])
assert featuresFrame["num_verbos"].isna().all()
assert not np.isinf(featuresFrame[anomalyColumns].to_numpy(dtype=float)).any()
assert measureText("casa " * 100, 300)["num_palavras"] == 60
assert measureText("casa " * 100, 300)["num_caracteres"] == 300
print("Contagens do prefixo verificadas; num_palavras não entra no treinamento.")

## 8. Análise estatística

Estatísticas descritivas por label, sem substituir vetores individuais por médias.
Esta inspeção do corpus completo atende ao protocolo exploratório: não usamos
seus resultados para selecionar features, ajustar hiperparâmetros ou threshold.
Qualquer escolha futura guiada por estes resultados exige nova avaliação independente.
Correlações de Pearson com comprimento são mostradas no total e por classe para
evitar confundir efeitos de classe e de tamanho. NaN em correlação pode indicar
feature constante. Mantemos as seis features recalculadas, inclusive possíveis redundâncias.

In [ ]:
statisticsRecords = []
for label, group in featuresFrame.groupby("label", sort=True):
    for feature in anomalyColumns:
        values = group[feature]
        q1, q3 = values.quantile([0.25, 0.75])
        statisticsRecords.append({
            "label": label, "class": labelNames[label], "feature": feature,
            "count": values.count(), "mean": values.mean(), "median": values.median(),
            "std": values.std(), "min": values.min(), "Q1": q1, "Q3": q3,
            "IQR": q3 - q1, "max": values.max(), "missingCount": values.isna().sum(),
        })
featureStatistics = pd.DataFrame(statisticsRecords).set_index(["label", "class", "feature"])
display(featureStatistics)

lengthColumns = ["num_palavras", "num_tokens"]
lengthCorrelations = pd.concat({
    name: group[anomalyColumns + lengthColumns].corr().loc[anomalyColumns, lengthColumns]
    for name, group in [
        ("All", featuresFrame),
        ("True (0)", featuresFrame.loc[featuresFrame["label"].eq(0)]),
        ("Fake (1)", featuresFrame.loc[featuresFrame["label"].eq(1)]),
    ]
}, names=["group", "feature"])
display(lengthCorrelations)
featureCorrelations = featuresFrame[anomalyColumns].corr()
display(featureCorrelations.round(3))
display(lengthCorrelations.xs("typeTokenRatio", level="feature"))
print("Pearson TTR vs diversidade:", featureCorrelations.loc["typeTokenRatio", "diversidade"])

## 9. Train / Validation / Test

True: 60% treino, 20% validação, 20% teste. Fake: 50% validação, 50% teste.
Todas as divisões usam `random_state=42`. Verificamos IDs exclusivos e cobertura
integral do corpus. **Nenhuma notícia Fake participa de fit ou calibração.**

Limitação do protocolo solicitado: a divisão é por notícia, não por assunto,
fonte ou data. O corpus possui pares True/Fake com o mesmo número-base; IDs
`123t` e `123` são notícias distintas, mas podem tratar do mesmo assunto em
partições diferentes. IDs exclusivos não demonstram independência temática.

In [ ]:
normalFrame = featuresFrame.loc[featuresFrame["label"].eq(0)].copy()
fakeFrame = featuresFrame.loc[featuresFrame["label"].eq(1)].copy()
normalTrainFrame, normalHoldoutFrame = train_test_split(
    normalFrame, train_size=0.60, random_state=RANDOM_STATE,
)
normalValidationFrame, normalTestFrame = train_test_split(
    normalHoldoutFrame, test_size=0.50, random_state=RANDOM_STATE,
)
fakeValidationFrame, fakeTestFrame = train_test_split(
    fakeFrame, test_size=0.50, random_state=RANDOM_STATE,
)
partitions = {
    "normalTrain": normalTrainFrame, "normalValidation": normalValidationFrame,
    "normalTest": normalTestFrame, "fakeValidation": fakeValidationFrame,
    "fakeTest": fakeTestFrame,
}
for name, frame in partitions.items():
    assert not frame.empty and frame["id"].is_unique
    assert frame["label"].eq(0 if name.startswith("normal") else 1).all()
for (leftName, left), (rightName, right) in combinations(partitions.items(), 2):
    assert set(left["id"]).isdisjoint(right["id"]), f"IDs compartilhados: {leftName}/{rightName}"
assert set().union(*(set(frame["id"]) for frame in partitions.values())) == set(featuresFrame["id"])
assert sum(len(frame) for frame in partitions.values()) == len(featuresFrame)
assert normalTrainFrame["label"].eq(0).all()
display(pd.DataFrame([
    {"partition": name, "count": len(frame), "label": int(frame["label"].iloc[0])}
    for name, frame in partitions.items()
]).set_index("partition"))

## 10. Isolation Forest

Pipeline fixo: imputação pela mediana + 300 árvores, sem StandardScaler.
O único `.fit()` recebe exclusivamente `normalTrain`. Uma guarda rejeita Fake,
infinitos e features inteiramente ausentes no treino (evita remoção silenciosa
de colunas pelo imputer). Nem médias do corpus nem dados de validação/teste
participam do ajuste.

In [ ]:
def fitNormalOnly(pipeline, trainingFrame):
    if trainingFrame.empty or not trainingFrame["label"].eq(0).all():
        raise ValueError("Treino permitido apenas com notícias True (label == 0).")
    trainingFeatures = trainingFrame[anomalyColumns]
    if np.isinf(trainingFeatures.to_numpy(dtype=float)).any():
        raise ValueError("Features de treino contêm infinito.")
    emptyColumns = trainingFeatures.columns[trainingFeatures.isna().all()].tolist()
    if emptyColumns:
        raise ValueError(f"Features inteiramente ausentes em normalTrain: {emptyColumns}")
    return pipeline.fit(trainingFeatures)

In [ ]:
anomalyPipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("detector", IsolationForest(
        n_estimators=300, contamination="auto", random_state=42, n_jobs=-1,
    )),
])
fitNormalOnly(anomalyPipeline, normalTrainFrame)
assert list(anomalyPipeline.feature_names_in_) == anomalyColumns
assert anomalyPipeline["detector"].n_features_in_ == len(anomalyColumns)
np.testing.assert_allclose(
    anomalyPipeline["imputer"].statistics_, normalTrainFrame[anomalyColumns].median().to_numpy(),
)
print(f"Pipeline ajustado em {len(normalTrainFrame)} notícias True e 0 Fake.")
display(pd.Series(anomalyPipeline["imputer"].statistics_, index=anomalyColumns, name="normalTrainMedian").to_frame())

## 11. Anomaly scores

Invertemos `decision_function`: score menor = mais normal; maior = mais anômalo.
Scores podem ser negativos. Não são probabilidades e não são convertidos em
percentuais de falsidade. Fake validation é reservada para inspeção visual.

In [ ]:
def calculateAnomalyScore(pipeline, featuresFrame):
    return -pipeline.decision_function(featuresFrame)

In [ ]:
normalValidationScores = calculateAnomalyScore(anomalyPipeline, normalValidationFrame[anomalyColumns])
fakeValidationScores = calculateAnomalyScore(anomalyPipeline, fakeValidationFrame[anomalyColumns])
testFrame = pd.concat([normalTestFrame, fakeTestFrame], ignore_index=True)
testResultsFrame = testFrame[["id", "label"] + anomalyColumns].copy()
testResultsFrame["anomalyScore"] = calculateAnomalyScore(anomalyPipeline, testFrame[anomalyColumns])
assert np.isfinite(normalValidationScores).all()
assert np.isfinite(fakeValidationScores).all()
assert np.isfinite(testResultsFrame["anomalyScore"]).all()

## 12. Threshold

Regra fixa: **percentil 95 dos scores de normalValidation**. Aproximadamente 5%
das notícias True de validação ultrapassam esse valor; empates podem aumentar
a proporção porque usamos `>=`. A taxa no teste pode ser diferente. Nenhuma
Fake ou notícia de teste define o threshold; `contamination="auto"` não escolhe
este corte empírico. Não usamos `predict()` do detector.

In [ ]:
threshold = np.quantile(normalValidationScores, 0.95)
testResultsFrame["isAnomaly"] = testResultsFrame["anomalyScore"] >= threshold
normalValidationAnomalyRate = np.mean(normalValidationScores >= threshold)
print(f"Threshold (percentil 95 de normalValidation): {threshold:.6f}")
print(f"True de validação marcadas como anômalas: {normalValidationAnomalyRate:.2%}")
assert np.isfinite(threshold)
assert "isFake" not in testResultsFrame.columns
display(testResultsFrame[["id", "label", "anomalyScore", "isAnomaly"]].head())

## 13. Avaliação

Avaliação final em **normalTest + fakeTest**. Classe positiva = **Fake (1)**
somente para medir discriminação. Precision/Recall/F1 e matriz comparam o
rótulo externo com o sinal `isAnomaly`, sem transformá-lo em classificação factual.
ROC-AUC e Average Precision usam scores contínuos. Reportamos **Average Precision
(AP)** como resumo da curva PR, não a integral trapezoidal PR-AUC.
A prevalência de Fake no teste é o referencial de AP de uma ordenação aleatória;
ela não representa a prevalência real na web.

In [ ]:
scoreStatistics = testResultsFrame.groupby("label")["anomalyScore"].agg(["count", "mean", "median", "std", "min", "max"])
scoreStatistics.insert(0, "class", scoreStatistics.index.map(labelNames))
display(scoreStatistics)

testLabels = testResultsFrame["label"].to_numpy()
testScores = testResultsFrame["anomalyScore"].to_numpy()
testFlags = testResultsFrame["isAnomaly"].to_numpy(dtype=int)
confusionMatrix = confusion_matrix(testLabels, testFlags, labels=[0, 1])
trueNegative, falsePositive, falseNegative, truePositive = confusionMatrix.ravel()
falsePositiveRate = falsePositive / (trueNegative + falsePositive)
fakeDetectionRate = truePositive / (falseNegative + truePositive)
fakePrevalence = np.mean(testLabels == 1)
metrics = {
    "ROC-AUC": roc_auc_score(testLabels, testScores),
    "Average Precision (AP)": average_precision_score(testLabels, testScores),
    "Precision": precision_score(testLabels, testFlags, zero_division=0),
    "Recall": recall_score(testLabels, testFlags, zero_division=0),
    "F1": f1_score(testLabels, testFlags, zero_division=0),
    "True false positive rate": falsePositiveRate,
    "Fake detection rate": fakeDetectionRate,
    "Fake prevalence (AP baseline)": fakePrevalence,
}
display(pd.Series(metrics, name="value").to_frame())
print(f"Notícias Fake detectadas como anômalas: {fakeDetectionRate:.2%}")
print(f"Notícias True marcadas como anômalas (FPR): {falsePositiveRate:.2%}")
display(pd.DataFrame(
    confusionMatrix, index=["Actual True (0)", "Actual Fake (1)"],
    columns=["Within threshold", "Anomaly"],
))
assert confusionMatrix.sum() == len(testResultsFrame)
assert set(testResultsFrame["id"]) == set(normalTestFrame["id"]) | set(fakeTestFrame["id"])
assert np.isclose(fakeDetectionRate, metrics["Recall"])

## 14. Visualizações

True (0) em azul e Fake (1) em laranja. Histogramas de densidade usam intervalos
comuns porque o teste tem tamanhos de classe diferentes. O primeiro painel
mostra validação apenas para inspeção; todos os demais mostram o teste.
O threshold permanece o mesmo, escolhido exclusivamente em normalValidation.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
trueScores = testResultsFrame.loc[testResultsFrame["label"].eq(0), "anomalyScore"]
fakeScores = testResultsFrame.loc[testResultsFrame["label"].eq(1), "anomalyScore"]

for axis, groups, title in [
    (axes[0, 0], [normalValidationScores, fakeValidationScores], "Validação: inspeção dos scores"),
    (axes[0, 1], [trueScores, fakeScores], "Teste: distribuição dos scores"),
]:
    bins = np.histogram_bin_edges(np.concatenate(groups), bins=40)
    for values, name, color in zip(groups, ["True (0)", "Fake (1)"], ["tab:blue", "tab:orange"]):
        axis.hist(values, bins=bins, density=True, alpha=0.5, label=name, color=color)
    axis.axvline(threshold, color="black", linestyle="--", label=f"Threshold = {threshold:.3f}")
    axis.set(title=title, xlabel="anomalyScore (maior = mais anômalo)", ylabel="Densidade")
    axis.legend(fontsize=8)

boxes = axes[0, 2].boxplot([trueScores, fakeScores], patch_artist=True)
for box, color in zip(boxes["boxes"], ["tab:blue", "tab:orange"]):
    box.set_facecolor(color)
    box.set_alpha(0.5)
axes[0, 2].set_xticks([1, 2], ["True (0)", "Fake (1)"])
axes[0, 2].axhline(threshold, color="black", linestyle="--", label="Threshold")
axes[0, 2].set(title="Teste: boxplot por label", ylabel="anomalyScore")
axes[0, 2].legend(fontsize=8)

rocFpr, rocTpr, _ = roc_curve(testLabels, testScores, pos_label=1)
axes[1, 0].plot(rocFpr, rocTpr, label=f"ROC-AUC = {metrics['ROC-AUC']:.3f}")
axes[1, 0].plot([0, 1], [0, 1], "k--", label="Referência aleatória")
axes[1, 0].scatter([falsePositiveRate], [fakeDetectionRate], color="black", label="Threshold fixo")
axes[1, 0].set(title="Teste: ROC (positivo = Fake)", xlabel="False positive rate (True)", ylabel="True positive rate (Fake)")
axes[1, 0].legend(fontsize=8)

prPrecision, prRecall, _ = precision_recall_curve(testLabels, testScores, pos_label=1)
axes[1, 1].plot(prRecall, prPrecision, label=f"AP = {metrics['Average Precision (AP)']:.3f}")
axes[1, 1].axhline(fakePrevalence, color="black", linestyle="--", label=f"Prevalência Fake = {fakePrevalence:.3f}")
axes[1, 1].scatter([metrics["Recall"]], [metrics["Precision"]], color="black", label="Threshold fixo")
axes[1, 1].set(title="Teste: Precision-Recall (positivo = Fake)", xlabel="Recall", ylabel="Precision", ylim=(0, 1.05))
axes[1, 1].legend(fontsize=8)

ConfusionMatrixDisplay(confusionMatrix, display_labels=["True (0)", "Fake (1)"]).plot(
    ax=axes[1, 2], colorbar=False, cmap="Blues", values_format="d",
)
axes[1, 2].set_xticks([0, 1], ["Dentro do padrão", "Anomalia"])
axes[1, 2].set(title="Teste: matriz após threshold", xlabel="Sinal do detector", ylabel="Rótulo real")
axes[1, 2].grid(False)
plt.show()

## 15. Casos extremos

Exemplos exclusivamente do teste: cinco True com maior score, cinco Fake com
maior score e cinco Fake com menor score. Todas as seis features são apresentadas;
não são atribuições causais ou importâncias do Isolation Forest. NaN exibido
na tabela corresponde à feature antes da imputação, preservada para auditoria.

In [ ]:
exampleColumns = ["id", "label", "anomalyScore", "isAnomaly"] + anomalyColumns
extremeCases = {
    "5 True com maior anomalyScore": testResultsFrame.loc[testResultsFrame["label"].eq(0)].nlargest(5, "anomalyScore"),
    "5 Fake com maior anomalyScore": testResultsFrame.loc[testResultsFrame["label"].eq(1)].nlargest(5, "anomalyScore"),
    "5 Fake com menor anomalyScore": testResultsFrame.loc[testResultsFrame["label"].eq(1)].nsmallest(5, "anomalyScore"),
}
for title, examples in extremeCases.items():
    display(Markdown(f"**{title}**"))
    display(examples[exampleColumns].reset_index(drop=True))

## 16. Conclusão

O resumo abaixo é calculado nesta execução, sem ajuste posterior do modelo ou do corte. AUC resume a ordenação; recall e FPR resumem o corte escolhido. Nenhuma dessas métricas demonstra significância estatística ou validade factual.

In [ ]:
display(Markdown(f"""
Foram treinadas **{len(normalTrainFrame)} notícias True e nenhuma Fake**,
usando até **{CHARACTER_LIMIT} caracteres** e seis features recalculadas.
No teste: **{truePositive}/{len(fakeTestFrame)} Fake detectadas** e
**{falsePositive}/{len(normalTestFrame)} True sinalizadas**.
ROC-AUC: **{metrics['ROC-AUC']:.4f}**; recall **{fakeDetectionRate:.2%}**;
FPR **{falsePositiveRate:.2%}**. Corte: **{threshold:.6f}**, calibrado apenas em True.

num_palavras permanece na extração, mas está fora da matriz de treino.
Correlações por classe estão na seção 8. Caracteres fixos não fixam palavras;
o corte também remove conteúdo e pode cortar frases. Não ajustamos o corte pelo teste.
A comparação com o texto completo não faz parte deste notebook didático.
**Anomalia não comprova falsidade.**
"""))